# Ch 17 · Collision III: Convex Shapes with GJK, EPA and Multi-Contact

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/17-convex-collision-gjk-epa.ipynb)

Companion notebook for the chapter page `chapters/17-convex-collision-gjk-epa.html` ([page](../chapters/17-convex-collision-gjk-epa.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

Every code cell below is the same file as the matching block on the chapter page (`examples/17-convex-collision-gjk-epa/`). Examples 17.2-17.6 are NumPy teaching code that mirrors `collision_gjk.py` (17.4 and 17.6 also compare with MuJoCo Warp); the others run MuJoCo Warp. Budgets grow on a GPU (`ON_GPU`) and stay small on a CPU. CPU numbers are never GPU numbers.

## 1. Why one general convex path
Pairs marked CONVEX in `MJ_COLLISION_TABLE` (cylinder, ellipsoid, box-box, mesh, heightfield) go through `collision_convex.py::convex_narrowphase`, which builds one kernel per convex type pair present: support functions, GJK, EPA when needed, multi-contact, `write_contact`. There is no MPR in this code base.

In [ ]:
# Example 17.1 - ex1_convex_scene.py
"""Example 17.1 - three convex pairs that no closed-form collider handles.

A cylinder, an ellipsoid and a hexagonal-prism mesh each rest on their own fixed box. In
MJ_COLLISION_TABLE all three pairs (cylinder-box, ellipsoid-box, box-mesh) are CONVEX, so
collision_convex.py::convex_narrowphase builds one GJK/EPA kernel per pair type. We print the
path of every pair, the contacts per pair at t = 0 and after 0.4 s of simulation, and nacon.
Run: python ex1_convex_scene.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.collision_driver import MJ_COLLISION_TABLE  # private API

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 200

# hexagonal prism: 12 vertices (>= 10, so support() hill-climbs the mesh graph)
HEX = " ".join(f"{0.05 * np.cos(a):.5f} {0.05 * np.sin(a):.5f} {z}"
               for z in (-0.03, 0.03) for a in np.arange(6) * np.pi / 3)

CONVEX_SCENE = f"""
<mujoco model="convex_scene">
  <option timestep="0.002"/>
  <asset><mesh name="hexprism" vertex="{HEX}"/></asset>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="2 2 .1"/>
    <geom name="table_cyl" type="box" pos="-0.4 0 0.05" size="0.1 0.1 0.05"/>
    <geom name="table_ell" type="box" pos="0 0 0.05" size="0.1 0.1 0.05"/>
    <geom name="table_mesh" type="box" pos="0.4 0 0.05" size="0.1 0.1 0.05"/>
    <body name="cyl" pos="-0.4 0 0.149"><freejoint/>
      <geom name="cyl" type="cylinder" size="0.05 0.05"/></body>
    <body name="ell" pos="0 0 0.129"><freejoint/>
      <geom name="ell" type="ellipsoid" size="0.06 0.04 0.03"/></body>
    <body name="hex" pos="0.4 0 0.129"><freejoint/>
      <geom name="hex" type="mesh" mesh="hexprism"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(CONVEX_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=24, njmax=96)
names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(mjm.ngeom)]

print("pair path in MJ_COLLISION_TABLE:")
for a, b in [("cyl", "table_cyl"), ("ell", "table_ell"), ("hex", "table_mesh")]:
    t1, t2 = sorted([mjw.GeomType(mjm.geom_type[names.index(a)]),
                     mjw.GeomType(mjm.geom_type[names.index(b)])])
    print(f"  {a:4s} vs {b:10s}: ({t1.name}, {t2.name}) -> {MJ_COLLISION_TABLE[(t1, t2)].name}")


def per_pair(d):
    n = min(int(d.nacon.numpy()[0]), d.naconmax)
    geom = d.contact.geom.numpy()[:n]
    wid = d.contact.worldid.numpy()[:n]
    dist = d.contact.dist.numpy()[:n]
    out = {}
    for (g1, g2), w, dd in zip(geom, wid, dist):
        if w != 0:
            continue
        key = f"{names[g1]}-{names[g2]}"
        out.setdefault(key, []).append(round(float(dd), 4))
    return n, out


mjw.kinematics(m, d)
mjw.collision(m, d)
n0, pairs0 = per_pair(d)
print("t = 0.0 s, world 0 contacts per pair (geom[0]-geom[1]: dist list):")
for k, v in sorted(pairs0.items()):
    print(f"  {k:16s} {len(v)} contacts  dist = {v}")
print(f"nacon over {NWORLD} worlds = {n0}")

for _ in range(NSTEP):
    mjw.step(m, d)
wp.synchronize()
n1, pairs1 = per_pair(d)
print(f"t = {NSTEP * mjm.opt.timestep:.1f} s, world 0 contacts per pair:")
for k, v in sorted(pairs1.items()):
    print(f"  {k:16s} {len(v)} contacts")
print(f"nacon over {NWORLD} worlds = {n1}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 2. Support functions and the Minkowski difference
`support(geom, dir)` returns the point of a shape furthest along `dir`, then pushes it out by `0.5 * margin`. The Minkowski difference A - B contains the origin exactly when A and B overlap, and its support is `s_A(d) - s_B(-d)`.

In [ ]:
# Example 17.2 - ex2_support_numpy.py
"""Example 17.2 - support functions in NumPy, checked against brute force.

A support function s(dir) returns the point of a shape that lies furthest along dir. The four
NumPy functions below follow collision_gjk.py::support line by line (capsule, ellipsoid, box:
rotate dir into the geom frame, pick the extreme point, rotate back, add pos, then inflate by
margin/2; the sphere uses the world dir directly).
Each is checked against the "analytic extreme": the max of dot(x, dir) over 20k surface samples.
The shapes are built with MjSpec (Ch 2 §11), so their sizes are MuJoCo's geom_size convention.
Run: python ex2_support_numpy.py
"""
import mujoco
import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)


def rotate(q):
    mat = np.zeros(9)
    mujoco.mju_quat2Mat(mat, np.asarray(q, float))
    return mat.reshape(3, 3)


def support(gtype, size, pos, rot, margin, dir):
    """NumPy mirror of collision_gjk.py::support for sphere, capsule, ellipsoid and box."""
    dir = dir / np.linalg.norm(dir)
    if gtype == "sphere":                      # returns early: its own 0.5*margin term
        return pos + (size[0] + 0.5 * margin) * dir
    local = rot.T @ dir                        # local_dir = transpose(rot) @ dir
    if gtype == "box":
        res = np.sign(local) * size            # the corner with the right signs
    elif gtype == "capsule":
        res = local * size[0]                  # sphere of radius r ...
        res[2] += np.sign(local[2]) * size[1]  # ... moved to the end cap +-halflength
    elif gtype == "ellipsoid":
        res = local * size
        res = res / np.linalg.norm(res)        # normalize(dir * size) * size
        res = res * size
    point = rot @ res + pos
    return point + dir * (0.5 * margin)        # margin inflation, as at the end of support()


def surface_samples(gtype, size, n=20000):
    u = rng.normal(size=(n, 3))
    u /= np.linalg.norm(u, axis=1, keepdims=True)
    if gtype == "sphere":
        return size[0] * u
    if gtype == "ellipsoid":
        return u * size
    if gtype == "capsule":
        p = size[0] * u
        p[:, 2] += np.where(u[:, 2] >= 0, size[1], -size[1])   # two hemispheres
        t = rng.uniform(-size[1], size[1], n)                    # plus the side wall
        side = np.column_stack([size[0] * u[:, 0], size[0] * u[:, 1], t])
        side[:, :2] /= np.linalg.norm(u[:, :2], axis=1, keepdims=True)
        side[:, :2] *= size[0]
        return np.vstack([p, side])
    if gtype == "box":
        corners = np.array([[sx, sy, sz] for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)])
        return np.vstack([corners * size, rng.uniform(-1, 1, (n, 3)) * size])


spec = mujoco.MjSpec()
body = spec.worldbody.add_body(name="shapes")
shapes = {
    "sphere": body.add_geom(type=mujoco.mjtGeom.mjGEOM_SPHERE, size=[0.1, 0, 0]),
    "capsule": body.add_geom(type=mujoco.mjtGeom.mjGEOM_CAPSULE, size=[0.05, 0.1, 0]),
    "ellipsoid": body.add_geom(type=mujoco.mjtGeom.mjGEOM_ELLIPSOID, size=[0.12, 0.06, 0.03]),
    "box": body.add_geom(type=mujoco.mjtGeom.mjGEOM_BOX, size=[0.1, 0.05, 0.02]),
}
mjm = spec.compile()

pos = np.array([0.3, -0.2, 0.5])
rot = rotate([np.cos(0.3), np.sin(0.3) * 0.6, np.sin(0.3) * 0.8, 0.0])   # some rotation
dirs = rng.normal(size=(6, 3))
for gid, (name, _) in enumerate(shapes.items()):
    size = mjm.geom_size[gid].copy()
    pts = surface_samples(name, size) @ rot.T + pos
    errs = []
    for dvec in dirs:
        dn = dvec / np.linalg.norm(dvec)
        s = support(name, size, pos, rot, 0.0, dvec)
        best = (pts @ dn).max()                # brute-force extreme along dir
        errs.append(best - s @ dn)             # >= 0 up to sampling error
    print(f"{name:9s} size={size}  max |extreme - dot(s, dir)| over 6 dirs = "
          f"{np.abs(errs).max():.2e}")

# margin: the support point moves out by margin/2 along dir (each geom gets half)
s0 = support("box", mjm.geom_size[3], pos, rot, 0.0, dirs[0])
s1 = support("box", mjm.geom_size[3], pos, rot, 0.02, dirs[0])
print("box support, margin 0   :", s0)
print("box support, margin 0.02:", s1, " shift =", round(float(np.linalg.norm(s1 - s0)), 4))

# Minkowski difference A - B: its support along dir is s_A(dir) - s_B(-dir)
a = support("sphere", mjm.geom_size[0], np.zeros(3), np.eye(3), 0.0, np.array([1.0, 0, 0]))
b = support("box", mjm.geom_size[3], np.array([0.25, 0, 0]), np.eye(3), 0.0,
            -np.array([1.0, 0, 0]))
print("support of (sphere - box) along +x:", a - b, "(x < 0: the shapes are 0.05 apart)")

## 3. GJK in 2-D
The loop of `collision_gjk.py::gjk`: support point along `-x_k`, Frank-Wolfe duality-gap stop, distance sub-algorithm, drop vertices with weight 0, stop when the simplex is full.

In [ ]:
# Example 17.3 - ex3_gjk_2d.py
"""Example 17.3 - GJK in 2D (NumPy), shaped like collision_gjk.py::gjk.

The loop is the same as MJWarp's: start from x_k = c1 - c2, take the support point of the
Minkowski difference A - B along -x_k, stop on the Frank-Wolfe duality gap, run the
distance sub-algorithm (_S1D for a segment, a 2D _S2D for a triangle) to get barycentric
weights, drop the vertices whose weight is 0, and stop when the simplex is "full" (a triangle
in 2D, a tetrahedron in 3D) because then the origin is inside A - B: the shapes overlap.
Prints the simplex per iteration for a separated and an overlapping pair of polygons and checks
the distance against brute force.
Run: python ex3_gjk_2d.py [--dump-json PATH]
  --dump-json PATH writes polygons and per-iteration simplices for the page's GJK canvas.
"""
import argparse
import json

import numpy as np

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()


def support_poly(verts, dir):
    """Support of a convex polygon: the vertex with the largest dot(v, dir) (like a mesh)."""
    return verts[np.argmax(verts @ dir)]


def s1d(s1, s2):
    """Mirror of _S1D: weights of the origin's projection onto segment s1-s2."""
    diff = s2 - s1
    p_o = s2 - (s2 @ diff) / (diff @ diff) * diff        # _project_origin_line
    index = int(np.argmax(np.abs(s1 - s2)))              # axis with the largest shadow
    mu_max = s1[index] - s2[index]
    c1, c2 = p_o[index] - s2[index], s1[index] - p_o[index]
    if np.sign(mu_max) == np.sign(c1) == np.sign(c2) and c1 != 0 and c2 != 0:
        return np.array([c1 / mu_max, c2 / mu_max])
    return np.array([0.0, 1.0])                          # outside: keep the newer vertex


def cross2(a, b):
    return a[0] * b[1] - a[1] * b[0]


def s2d(s1, s2, s3):
    """2D analogue of _S2D: signed areas; inside -> weights, else the best edge via s1d."""
    m = cross2(s2 - s1, s3 - s1)                         # twice the signed area
    c = np.array([cross2(s2, s3), cross2(s3, s1), cross2(s1, s2)])   # areas with origin
    if np.all(np.sign(c) == np.sign(m)) and m != 0:
        return c / m, True                               # origin inside the triangle
    best, lam = np.inf, None
    for i, (a, b) in enumerate([(1, 2), (0, 2), (0, 1)]):
        if np.sign(c[i]) == np.sign(m):
            continue                                     # origin is on the inner side
        sub = s1d([s1, s2, s3][a], [s1, s2, s3][b])
        x = sub[0] * [s1, s2, s3][a] + sub[1] * [s1, s2, s3][b]
        if x @ x < best:
            best, lam = x @ x, np.zeros(3)
            lam[a], lam[b] = sub
    return lam, False


def gjk2d(A, B, iterations=20, tol=1e-6):
    """Returns (dist, overlap, log). log has one entry per iteration."""
    x_k = A.mean(0) - B.mean(0)                          # x1_0 - x2_0 (centres)
    simplex, log = [], []
    eps = 0.5 * tol * tol
    for k in range(iterations):
        xnorm = np.linalg.norm(x_k)
        if xnorm < tol:
            break
        d = x_k / xnorm
        s = support_poly(A, -d) - support_poly(B, d)     # support of A - B along -x_k
        gap = x_k @ (x_k - s)                            # Frank-Wolfe duality gap
        if gap < eps:
            log.append({"k": k, "x": x_k.tolist(), "s": s.tolist(), "gap": gap,
                        "simplex": [p.tolist() for p in simplex], "event": "gap"})
            break
        simplex.append(s)
        if len(simplex) == 1:
            lam, inside = np.array([1.0]), False
        elif len(simplex) == 2:
            lam, inside = s1d(*simplex), False
        else:
            lam, inside = s2d(*simplex)
        keep = [i for i in range(len(simplex)) if lam[i] != 0.0]
        simplex = [simplex[i] for i in keep]
        lam = lam[keep]
        x_k = sum(l * p for l, p in zip(lam, simplex))
        log.append({"k": k, "x": x_k.tolist(), "s": s.tolist(), "gap": gap,
                    "simplex": [p.tolist() for p in simplex],
                    "event": "full" if inside else "reduce"})
        if inside:                                       # 2D "tetrahedron": origin enclosed
            return 0.0, True, log
    return float(np.linalg.norm(x_k)), False, log


def brute_distance(A, B):
    def pt_seg(p, a, b):
        t = np.clip((p - a) @ (b - a) / ((b - a) @ (b - a)), 0, 1)
        return np.linalg.norm(p - (a + t * (b - a)))
    best = np.inf
    for P, Q in [(A, B), (B, A)]:
        for p in P:
            for i in range(len(Q)):
                best = min(best, pt_seg(p, Q[i], Q[(i + 1) % len(Q)]))
    return best


A = np.array([[0.0, 0.0], [1.0, -0.3], [1.6, 0.4], [1.0, 1.1], [0.1, 0.9]])
B0 = np.array([[0.0, 0.0], [0.9, 0.1], [1.0, 0.8], [0.2, 0.9]])
t = np.radians(35.0)
B0 = B0 @ np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]]).T   # B turned by 35 deg
cases = {"separated": B0 + [1.9, 0.9], "overlapping": B0 + [1.8, -0.6]}
dump = {"A": A.tolist(), "cases": {}}
for name, B in cases.items():
    dist, overlap, log = gjk2d(A, B)
    print(f"--- {name}")
    for e in log:
        simp = " ".join(f"({p[0]:.3f},{p[1]:.3f})" for p in e["simplex"])
        print(f"  k={e['k']}  {e['event']:6s}  n={len(e['simplex'])}  "
              f"|x|={np.linalg.norm(e['x']):.4f}  gap={e['gap']:.4f}  simplex: {simp}")
    print(f"  GJK distance = {dist:.4f}  overlap = {overlap}  brute force = "
          f"{0.0 if overlap else brute_distance(A, B):.4f}")
    dump["cases"][name] = {"B": B.tolist(), "dist": dist, "overlap": overlap,
                           "log": [{**e, "gap": round(e["gap"], 5)} for e in log]}

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

### Interactive GJK stepping
Pick a case and an iteration (needs `ipywidgets`, which Colab has; otherwise it draws the overlapping case iteration by iteration).

In [ ]:
# Interactive GJK stepping (uses gjk2d, A and cases from Example 17.3: run that cell first).
%matplotlib inline
import numpy as np
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("skipped: matplotlib not installed")


def hull(P):
    P = sorted(map(tuple, P))
    def cr(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
    lo, up = [], []
    for p in P:
        while len(lo) >= 2 and cr(lo[-2], lo[-1], p) <= 0:
            lo.pop()
        lo.append(p)
    for p in reversed(P):
        while len(up) >= 2 and cr(up[-2], up[-1], p) <= 0:
            up.pop()
        up.append(p)
    return np.array(lo[:-1] + up[:-1])


def show(case="separated", k=0):
    B = cases[case]
    dist, overlap, log = gjk2d(A, B)
    k = min(k, len(log) - 1)
    M = hull([a - b for a in A for b in B])
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
    for P, c in ((A, "C0"), (B, "C1")):
        ax[0].fill(*np.vstack([P, P[:1]]).T, alpha=0.35, color=c)
    ax[0].set_title("world: A and B"); ax[0].set_aspect("equal")
    ax[1].fill(*np.vstack([M, M[:1]]).T, alpha=0.25, color="C9")
    ax[1].plot(0, 0, "k+", ms=14, mew=2)
    e = log[k]
    S = np.array(e["simplex"])
    if len(S):
        ax[1].fill(*np.vstack([S, S[:1]]).T, alpha=0.5, color="C2", lw=3, edgecolor="C2")
    ax[1].plot(*e["s"], "o", color="C1", ms=9, label="new support point")
    ax[1].plot([0, e["x"][0]], [0, e["x"][1]], "r-", lw=2, label="x_k")
    ax[1].set_title(f"A - B, k={e['k']} ({e['event']}), |x|={np.linalg.norm(e['x']):.4f}")
    ax[1].set_aspect("equal"); ax[1].legend(loc="lower left")
    plt.show()


if plt is not None:
    try:
        from ipywidgets import interact
        interact(show, case=list(cases), k=(0, 5))
    except ImportError:
        for kk in range(4):
            show("overlapping", kk)

### Exercise 17.1: witness points
Extend the 2-D GJK so it also returns `x1` on A and `x2` on B (the same `lmbda` applied to the per-shape support points, like `simplex1`/`simplex2` in `gjk()`). Run it right after Example 17.3: later examples redefine `s1d`, `s2d` and `A` in 3-D.

In [ ]:
# Exercise 17.1 (needs support_poly, s1d, s2d, brute_distance and A from Example 17.3: run it first)
import numpy as np


def gjk2d_witness(A, B, iterations=20, tol=1e-6):
    """Return (dist, x1, x2): dist = |x1 - x2|, x1 on A, x2 on B."""
    ...   # keep simplex1 / simplex2 next to simplex and drop the same vertices


def cross2(a, b):
    return a[0] * b[1] - a[1] * b[0]


def on_or_in(P, x, eps=1e-9):
    return all(cross2(P[(i + 1) % len(P)] - P[i], x - P[i]) >= -eps for i in range(len(P)))


B0 = np.array([[0.0, 0.0], [0.9, 0.1], [1.0, 0.8], [0.2, 0.9]])
if gjk2d_witness(A, B0 + [1.9, 0.9]) is None:
    print("Exercise 17.1: fill in gjk2d_witness, then re-run this cell")
else:
    for off in ([1.9, 0.9], [2.5, 0.0], [0.4, 1.6]):
        B = B0 + off
        dist, x1, x2 = gjk2d_witness(A, B)
        assert abs(dist - brute_distance(A, B)) < 1e-6
        assert abs(np.linalg.norm(x1 - x2) - dist) < 1e-9
        assert on_or_in(A, x1) and on_or_in(B, x2)
    print("all 3 cases pass")

In [ ]:
#@title Exercise 17.1: solution (double-click to show the code)
def gjk2d_witness(A, B, iterations=20, tol=1e-6):
    x_k = A.mean(0) - B.mean(0)
    simplex, simplex1, simplex2 = [], [], []
    lam = np.ones(1)
    for k in range(iterations):
        xnorm = np.linalg.norm(x_k)
        if xnorm < tol:
            break
        d = x_k / xnorm
        p1, p2 = support_poly(A, -d), support_poly(B, d)
        s = p1 - p2
        if x_k @ (x_k - s) < 0.5 * tol * tol:
            break
        simplex.append(s); simplex1.append(p1); simplex2.append(p2)
        if len(simplex) == 1:
            lam, inside = np.array([1.0]), False
        elif len(simplex) == 2:
            lam, inside = s1d(*simplex), False
        else:
            lam, inside = s2d(*simplex)
        keep = [i for i in range(len(simplex)) if lam[i] != 0.0]
        simplex = [simplex[i] for i in keep]
        simplex1 = [simplex1[i] for i in keep]
        simplex2 = [simplex2[i] for i in keep]
        lam = lam[keep]
        x_k = sum(l * p for l, p in zip(lam, simplex))
        if inside:
            break
    x1 = sum(l * p for l, p in zip(lam, simplex1))   # same weights, applied to each shape
    x2 = sum(l * p for l, p in zip(lam, simplex2))
    return float(np.linalg.norm(x_k)), x1, x2


def cross2(a, b):
    return a[0] * b[1] - a[1] * b[0]


def on_or_in(P, x, eps=1e-9):
    return all(cross2(P[(i + 1) % len(P)] - P[i], x - P[i]) >= -eps for i in range(len(P)))


B0 = np.array([[0.0, 0.0], [0.9, 0.1], [1.0, 0.8], [0.2, 0.9]])
for off in ([1.9, 0.9], [2.5, 0.0], [0.4, 1.6]):
    B = B0 + off
    dist, x1, x2 = gjk2d_witness(A, B)
    assert abs(dist - brute_distance(A, B)) < 1e-6
    assert abs(np.linalg.norm(x1 - x2) - dist) < 1e-9
    assert on_or_in(A, x1) and on_or_in(B, x2)
    print(f"offset {off}: dist = {dist:.4f}  x1 = {x1.round(4)}  x2 = {x2.round(4)}")
print("all 3 cases pass")

## 3b. GJK in 3-D with signed volumes
`_S3D` uses cofactors of the 4x4 matrix [[s1 s2 s3 s4], [1 1 1 1]]; all with the sign of their sum means the origin is inside. Otherwise `_S2D` on the faces with the wrong sign, and `_S1D` on edges.

In [ ]:
# Example 17.4 - ex4_gjk_3d.py
"""Example 17.4 - GJK in 3D with the signed-volume sub-algorithm, checked against MJWarp.

s1d / s2d / s3d below are line-by-line NumPy ports of collision_gjk.py::_S1D, _S2D and _S3D
(signed volumes and cofactors, no Voronoi-region tests). The loop is collision_gjk.py::gjk.
Two boxes are tested: (w0) separated by about 2 cm with margin 0.05, so MJWarp writes contacts
whose dist is the true gap, and (w1) overlapping, where GJK ends with a tetrahedron (n == 4).
For each iteration we print which sub-simplex _S3D/_S2D/_S1D kept (vertex, edge, face or the
whole tetrahedron) and finally compare the GJK distance with d.contact.dist from MJWarp and
with MuJoCo C.
Run: python ex4_gjk_3d.py [--dump-json PATH]
  --dump-json PATH writes the per-iteration simplices for the page's 3-D GJK view.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)
parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

FLOAT_MAX, MINVAL = 1e30, 1e-15


def det3(a, b, c):
    return a @ np.cross(b, c)


def same_sign(a, b):
    return (a > 0 and b > 0) or (a < 0 and b < 0)


def project_origin_line(v1, v2):
    diff = v2 - v1
    return v2 - (v2 @ diff) / (diff @ diff) * diff


def project_origin_plane(v1, v2, v3):
    for a, b, c in [(v3 - v2, v2 - v1, v2), (v2 - v1, v3 - v1, v1), (v3 - v1, v3 - v2, v3)]:
        n = np.cross(a, b)
        nv, nn = n @ c, n @ n
        if nn == 0.0:
            return None
        if nv != 0.0 and nn > MINVAL:
            return (nv / nn) * n
    return (nv / nn) * n


def s1d(s1, s2):
    p_o = project_origin_line(s1, s2)
    index = 0
    mu_max = s1[0] - s2[0]
    for i in (1, 2):
        if abs(s1[i] - s2[i]) >= abs(mu_max):
            mu_max, index = s1[i] - s2[i], i
    c1, c2 = p_o[index] - s2[index], s1[index] - p_o[index]
    if same_sign(mu_max, c1) and same_sign(mu_max, c2):
        return np.array([c1 / mu_max, c2 / mu_max])
    return np.array([0.0, 1.0])


def s2d(s1, s2, s3):
    p_o = project_origin_plane(s1, s2, s3)
    if p_o is None:
        return np.append(s1d(s1, s2), 0.0)
    # minors M_14, M_24, M_34 of [[s1 s2 s3]; [1 1 1]] (same formulas as _S2D)
    m14 = (s2[1] * s3[2] - s2[2] * s3[1] - s1[1] * s3[2] + s1[2] * s3[1]
           + s1[1] * s2[2] - s1[2] * s2[1])
    m24 = (s2[0] * s3[2] - s2[2] * s3[0] - s1[0] * s3[2] + s1[2] * s3[0]
           + s1[0] * s2[2] - s1[2] * s2[0])
    m34 = (s2[0] * s3[1] - s2[1] * s3[0] - s1[0] * s3[1] + s1[1] * s3[0]
           + s1[0] * s2[1] - s1[1] * s2[0])
    # drop the axis with the largest projection of the triangle
    if abs(m14) >= abs(m24) and abs(m14) >= abs(m34):
        m_max, keep = m14, [1, 2]
    elif abs(m24) >= abs(m34):
        m_max, keep = m24, [0, 2]
    else:
        m_max, keep = m34, [0, 1]
    a, b, c, p = (v[keep] for v in (s1, s2, s3, p_o))
    # cofactors C31, C32, C33 = signed areas of (p, s2, s3), (p, s3, s1), (p, s1, s2)
    c31 = p[0] * b[1] + p[1] * c[0] + b[0] * c[1] - p[0] * c[1] - p[1] * b[0] - c[0] * b[1]
    c32 = p[0] * c[1] + p[1] * a[0] + c[0] * a[1] - p[0] * a[1] - p[1] * c[0] - a[0] * c[1]
    c33 = p[0] * a[1] + p[1] * b[0] + a[0] * b[1] - p[0] * b[1] - p[1] * a[0] - b[0] * a[1]
    comp = [same_sign(m_max, c31), same_sign(m_max, c32), same_sign(m_max, c33)]
    if all(comp):
        return np.array([c31, c32, c33]) / m_max         # projection inside the triangle
    dmin, lam = FLOAT_MAX, np.zeros(3)
    for i, (u, v) in enumerate([(1, 2), (0, 2), (0, 1)]):
        if comp[i]:
            continue
        sub = s1d([s1, s2, s3][u], [s1, s2, s3][v])
        x = sub[0] * [s1, s2, s3][u] + sub[1] * [s1, s2, s3][v]
        if x @ x < dmin:
            dmin, lam = x @ x, np.zeros(3)
            lam[u], lam[v] = sub
    return lam


def s3d(s1, s2, s3, s4):
    c41, c42, c43, c44 = -det3(s2, s3, s4), det3(s1, s3, s4), -det3(s1, s2, s4), det3(s1, s2, s3)
    m_det = c41 + c42 + c43 + c44                        # 6 x signed volume
    comp = [same_sign(m_det, c) for c in (c41, c42, c43, c44)]
    if all(comp):
        return np.array([c41, c42, c43, c44]) / m_det    # origin inside: keep all four
    S, dmin, lam = [s1, s2, s3, s4], FLOAT_MAX, np.zeros(4)
    for drop in range(4):
        if comp[drop]:
            continue
        idx = [i for i in range(4) if i != drop]
        sub = s2d(*(S[i] for i in idx))
        x = sum(sub[j] * S[idx[j]] for j in range(3))
        if x @ x < dmin:
            dmin, lam = x @ x, np.zeros(4)
            lam[idx] = sub
    return lam


def box_support(pos, rot, size, dir):
    return rot @ (np.sign(rot.T @ dir) * size) + pos


def gjk(g1, g2, iterations=35):
    """g = (pos, rot, size) of a box. Discrete geoms: tolerance ignored, as in MJWarp."""
    x_k = g1[0] - g2[0]
    simplex, log = [], []
    xnorm, xnorm_prev = np.linalg.norm(x_k), 0.0
    names = {1: "vertex", 2: "edge", 3: "face", 4: "tetrahedron (origin inside)"}
    for k in range(iterations):
        if xnorm < MINVAL or abs(xnorm_prev - xnorm) < MINVAL:
            break
        d = x_k / xnorm
        s = box_support(*g1, -d) - box_support(*g2, d)
        if x_k @ (x_k - s) < 0.0:                        # epsilon = 0 for discrete geoms
            break
        simplex.append(s)
        n = len(simplex)
        lam = {1: lambda: np.ones(1), 2: lambda: s1d(*simplex), 3: lambda: s2d(*simplex),
               4: lambda: s3d(*simplex)}[n]()
        kept = [i for i in range(n) if lam[i] != 0.0]
        simplex = [simplex[i] for i in kept]
        x_k = sum(lam[i] * p for i, p in zip(kept, simplex))
        xnorm_prev, xnorm = (0.0 if k == 0 else xnorm), np.linalg.norm(x_k)
        log.append({"k": k, "n_in": n, "kept": kept, "region": names[len(simplex)],
                    "s": s.round(4).tolist(), "x": x_k.round(5).tolist(),
                    "simplex": [p.round(4).tolist() for p in simplex]})
        if len(simplex) == 4:
            return 0.0, log
    return float(xnorm), log


def quat_z(deg):
    a = np.radians(deg) / 2
    return np.array([np.cos(a), 0, 0, np.sin(a)])


TWO_BOXES = """
<mujoco model="two_boxes">
  <worldbody>
    <geom name="A" type="box" size="0.1 0.1 0.1"/>
    <body name="B" pos="0.3 0 0"><freejoint/>
      <geom name="B" type="box" size="0.1 0.1 0.1" margin="0.05"/></body>
  </worldbody>
</mujoco>
"""
CASES = [("w0 separated", [0.1 + 0.1 * np.sqrt(2) + 0.02, 0.03, 0.0], 45.0),   # 2 cm gap
         ("w1 overlapping", [0.17, 0.04, 0.03], 30.0)]

mjm = mujoco.MjModel.from_xml_string(TWO_BOXES)
mjd = mujoco.MjData(mjm)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=len(CASES), nconmax=8, njmax=32)
qpos = np.tile(mjm.qpos0, (len(CASES), 1))
for w, (_, p, deg) in enumerate(CASES):
    qpos[w, :3], qpos[w, 3:7] = p, quat_z(deg)
d.qpos.assign(qpos.astype(np.float32))
mjw.kinematics(m, d)
mjw.collision(m, d)
n = min(int(d.nacon.numpy()[0]), d.naconmax)
cw, cdist = d.contact.worldid.numpy()[:n], d.contact.dist.numpy()[:n]

dump = {"cases": []}
for w, (name, p, deg) in enumerate(CASES):
    rot = np.zeros(9)
    mujoco.mju_quat2Mat(rot, quat_z(deg))
    gA = (np.zeros(3), np.eye(3), np.full(3, 0.1))
    gB = (np.array(p), rot.reshape(3, 3), np.full(3, 0.1))
    dist, log = gjk(gA, gB)
    print(f"--- {name} (B at {np.round(p, 4).tolist()}, turned {deg} deg about z)")
    for e in log:
        print(f"  k={e['k']}  {e['n_in']} pts in -> kept {e['kept']} = {e['region']:9s}"
              f"  |x| = {np.linalg.norm(e['x']):.5f}")
    mjd.qpos[:] = qpos[w]
    mujoco.mj_forward(mjm, mjd)
    c_dist = sorted(round(float(mjd.contact.dist[i]), 5) for i in range(mjd.ncon))
    print(f"  teaching GJK dist = {dist:.5f}")
    print(f"  MJWarp contact.dist = {[round(float(v), 5) for v in np.sort(cdist[cw == w])]}")
    print(f"  MuJoCo C contact.dist = {c_dist}")
    dump["cases"].append({"name": name, "B": np.round(p, 4).tolist(), "deg": deg, "dist": dist,
                          "log": log})
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 4. EPA
Grow a polytope inside A - B: closest face (lower bound), support point along its normal (upper bound), delete visible faces, attach new faces to the horizon (`_add_edge`).

In [ ]:
# Example 17.5 - ex5_epa_2d.py
"""Example 17.5 - EPA in 2D (NumPy), starting from the triangle GJK ended with.

GJK only says "the origin is inside A - B". EPA (collision_gjk.py::_epa) recovers HOW deep:
it grows a polytope inside A - B until its face closest to the origin lies on the boundary.
Each iteration: take the closest face (lower bound on the depth), get the support point w along
its normal (upper bound = dot(n, w)), stop when upper - lower < eps; otherwise delete every face
that w can see, keep the horizon (in 2D: the two end vertices of the deleted chain) and attach
new faces from w to the horizon. Prints depth and normal per iteration and checks the result
against a brute-force separating-axis search.
Run: python ex5_epa_2d.py [--dump-json PATH]
  --dump-json PATH writes the polytope per iteration for the page's 2-D EPA stepper.
"""
import argparse
import json

import numpy as np

np.set_printoptions(precision=4, suppress=True)
parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()


def cross2(a, b):
    return a[0] * b[1] - a[1] * b[0]


def support_poly(verts, dir):
    return verts[np.argmax(verts @ dir)]


def mink_support(A, B, dir):
    return support_poly(A, dir) - support_poly(B, -dir)          # support of A - B along dir


def gjk_triangle(A, B, iterations=20):
    """Compact 2D GJK (Example 17.3) that returns the final simplex when it encloses 0."""
    x = A.mean(0) - B.mean(0)
    simplex = []
    for _ in range(iterations):
        s = mink_support(A, B, -x / np.linalg.norm(x))
        if x @ (x - s) < 1e-12:
            return None                                          # converged: separated
        simplex.append(s)
        if len(simplex) == 3:
            P = np.array(simplex)
            area = [cross2(P[(i + 1) % 3] - P[i], -P[i]) for i in range(3)]
            if np.all(np.sign(area) == np.sign(area[0])):
                return P                                         # origin inside
        # keep the closest feature to the origin (brute force is fine for a teaching code)
        best = None
        for idx in ([0], [1], [2], [0, 1], [0, 2], [1, 2]):
            if max(idx) >= len(simplex):
                continue
            pts = [simplex[i] for i in idx]
            if len(pts) == 1:
                q = pts[0]
            else:
                e = pts[1] - pts[0]
                q = pts[0] + np.clip(-(pts[0] @ e) / (e @ e), 0, 1) * e
            if best is None or q @ q < best[0] - 1e-15:
                best = (q @ q, idx, q)
        simplex = [simplex[i] for i in best[1]]
        x = best[2]
    return None


def edge_info(p, q, center):
    """Outward unit normal of edge pq and the origin's distance to its line (face_pr)."""
    e = q - p
    n = np.array([e[1], -e[0]])
    n /= np.linalg.norm(n)
    if n @ (p - center) < 0:
        n = -n
    return n, float(n @ p)                                       # outward normal, distance


def epa2d(A, B, P, iterations=35, tol=1e-6):
    center = P.mean(0)
    verts = [v for v in P]
    if cross2(verts[1] - verts[0], verts[2] - verts[0]) < 0:
        verts = verts[::-1]                                      # keep the polygon ccw
    upper, log = np.inf, []
    for k in range(iterations):
        faces = [(i, (i + 1) % len(verts)) for i in range(len(verts))]
        info = [edge_info(verts[i], verts[j], center) for i, j in faces]
        idx = int(np.argmin([dist for _, dist in info]))        # closest face: lower bound
        n, lower = info[idx]
        w = mink_support(A, B, n)
        upper = min(upper, float(n @ w))                         # upper bound on the depth
        entry = {"k": k, "verts": [v.round(4).tolist() for v in verts], "face": idx,
                 "lower": round(lower, 5), "upper": round(upper, 5), "n": n.round(4).tolist(),
                 "w": w.round(4).tolist()}
        if upper - lower < tol:
            entry["event"] = "converged"
            log.append(entry)
            return lower, n, log
        # delete every face that w can see; the horizon is where visible meets hidden
        visible = [f for f, (nf, df) in enumerate(info) if nf @ w - df > 1e-10]
        entry["visible"], entry["event"] = visible, "expand"
        log.append(entry)
        nf = len(faces)
        first = [f for f in visible if (f - 1) % nf not in visible][0]  # chain is contiguous
        start = faces[first][0]                                  # horizon vertex 1
        end = faces[(first + len(visible) - 1) % nf][1]          # horizon vertex 2
        new = [verts[start], w]
        i = end
        while i != start:                                        # keep the hidden chain
            new.append(verts[i])
            i = (i + 1) % len(verts)
        verts = new
    return lower, n, log


def sat_depth(A, B):
    """Brute force: the smallest overlap over all edge normals of both polygons."""
    best = (np.inf, None)
    for P in (A, B):
        for i in range(len(P)):
            e = P[(i + 1) % len(P)] - P[i]
            n = np.array([e[1], -e[0]]) / np.linalg.norm(e)
            overlap = min((A @ n).max() - (B @ n).min(), (B @ n).max() - (A @ n).min())
            if overlap < best[0]:
                best = (overlap, n)
    return best[0]


A = np.array([[0.0, 0.0], [1.0, -0.3], [1.6, 0.4], [1.0, 1.1], [0.1, 0.9]])
B0 = np.array([[0.0, 0.0], [0.9, 0.1], [1.0, 0.8], [0.2, 0.9]])
t = np.radians(35.0)
B0 = B0 @ np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]]).T
dump = {"A": A.tolist(), "cases": {}}
for name, off in [("shallow", [1.8, -0.6]), ("deep", [1.2, -0.2])]:
    B = B0 + off
    P = gjk_triangle(A, B)
    depth, n, log = epa2d(A, B, P)
    print(f"--- {name}: GJK triangle {P.round(3).tolist()}")
    for e in log:
        print(f"  k={e['k']}  nvert={len(e['verts'])}  lower={e['lower']:.5f}  "
              f"upper={e['upper']:.5f}  n={e['n']}  {e['event']}"
              + (f" (deleted faces {e['visible']})" if e["event"] == "expand" else ""))
    print(f"  EPA depth = {depth:.5f}  normal = {n.round(4)}   brute-force SAT depth = "
          f"{sat_depth(A, B):.5f}")
    dump["cases"][name] = {"B": B.round(4).tolist(), "depth": round(depth, 5), "log": log}

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Example 17.6 - ex6_mjw_vs_teaching.py
"""Example 17.6 - EPA in 3D (NumPy) vs MJWarp's box-box penetration depth.

The teaching EPA mirrors collision_gjk.py::_epa: faces store the origin's projection onto
their plane (face_pr, pointing outward, see _attach_face) and its squared length (face_norm2);
each iteration picks the closest valid face, adds the support point w along its normal,
deletes every face that w can see, collects the horizon with _add_edge semantics (an edge
added twice is shared by two deleted faces and cancels) and attaches one new face per horizon
edge. GJK (the signed-volume code of Example 17.4) supplies the start tetrahedron, as in
_polytope4. Two axis-aligned boxes overlap by 1 cm; we compare depth and normal with MJWarp.
Run: python ex6_mjw_vs_teaching.py [--dump-json PATH]
  --dump-json PATH writes every EPA iteration for the page's 3-D EPA stepper.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)
parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

# ---- GJK, copied from Example 17.4 (signed-volume sub-algorithm) ----
FLOAT_MAX, MINVAL = 1e30, 1e-15


def det3(a, b, c):
    return a @ np.cross(b, c)


def same_sign(a, b):
    return (a > 0 and b > 0) or (a < 0 and b < 0)


def project_origin_line(v1, v2):
    diff = v2 - v1
    return v2 - (v2 @ diff) / (diff @ diff) * diff


def project_origin_plane(v1, v2, v3):
    for a, b, c in [(v3 - v2, v2 - v1, v2), (v2 - v1, v3 - v1, v1), (v3 - v1, v3 - v2, v3)]:
        n = np.cross(a, b)
        nv, nn = n @ c, n @ n
        if nn == 0.0:
            return None
        if nv != 0.0 and nn > MINVAL:
            return (nv / nn) * n
    return (nv / nn) * n


def s1d(s1, s2):
    p_o = project_origin_line(s1, s2)
    index = 0
    mu_max = s1[0] - s2[0]
    for i in (1, 2):
        if abs(s1[i] - s2[i]) >= abs(mu_max):
            mu_max, index = s1[i] - s2[i], i
    c1, c2 = p_o[index] - s2[index], s1[index] - p_o[index]
    if same_sign(mu_max, c1) and same_sign(mu_max, c2):
        return np.array([c1 / mu_max, c2 / mu_max])
    return np.array([0.0, 1.0])


def s2d(s1, s2, s3):
    p_o = project_origin_plane(s1, s2, s3)
    if p_o is None:
        return np.append(s1d(s1, s2), 0.0)
    # minors M_14, M_24, M_34 of [[s1 s2 s3]; [1 1 1]] (same formulas as _S2D)
    m14 = (s2[1] * s3[2] - s2[2] * s3[1] - s1[1] * s3[2] + s1[2] * s3[1]
           + s1[1] * s2[2] - s1[2] * s2[1])
    m24 = (s2[0] * s3[2] - s2[2] * s3[0] - s1[0] * s3[2] + s1[2] * s3[0]
           + s1[0] * s2[2] - s1[2] * s2[0])
    m34 = (s2[0] * s3[1] - s2[1] * s3[0] - s1[0] * s3[1] + s1[1] * s3[0]
           + s1[0] * s2[1] - s1[1] * s2[0])
    # drop the axis with the largest projection of the triangle
    if abs(m14) >= abs(m24) and abs(m14) >= abs(m34):
        m_max, keep = m14, [1, 2]
    elif abs(m24) >= abs(m34):
        m_max, keep = m24, [0, 2]
    else:
        m_max, keep = m34, [0, 1]
    a, b, c, p = (v[keep] for v in (s1, s2, s3, p_o))
    # cofactors C31, C32, C33 = signed areas of (p, s2, s3), (p, s3, s1), (p, s1, s2)
    c31 = p[0] * b[1] + p[1] * c[0] + b[0] * c[1] - p[0] * c[1] - p[1] * b[0] - c[0] * b[1]
    c32 = p[0] * c[1] + p[1] * a[0] + c[0] * a[1] - p[0] * a[1] - p[1] * c[0] - a[0] * c[1]
    c33 = p[0] * a[1] + p[1] * b[0] + a[0] * b[1] - p[0] * b[1] - p[1] * a[0] - b[0] * a[1]
    comp = [same_sign(m_max, c31), same_sign(m_max, c32), same_sign(m_max, c33)]
    if all(comp):
        return np.array([c31, c32, c33]) / m_max         # projection inside the triangle
    dmin, lam = FLOAT_MAX, np.zeros(3)
    for i, (u, v) in enumerate([(1, 2), (0, 2), (0, 1)]):
        if comp[i]:
            continue
        sub = s1d([s1, s2, s3][u], [s1, s2, s3][v])
        x = sub[0] * [s1, s2, s3][u] + sub[1] * [s1, s2, s3][v]
        if x @ x < dmin:
            dmin, lam = x @ x, np.zeros(3)
            lam[u], lam[v] = sub
    return lam


def s3d(s1, s2, s3, s4):
    c41, c42, c43, c44 = -det3(s2, s3, s4), det3(s1, s3, s4), -det3(s1, s2, s4), det3(s1, s2, s3)
    m_det = c41 + c42 + c43 + c44                        # 6 x signed volume
    comp = [same_sign(m_det, c) for c in (c41, c42, c43, c44)]
    if all(comp):
        return np.array([c41, c42, c43, c44]) / m_det    # origin inside: keep all four
    S, dmin, lam = [s1, s2, s3, s4], FLOAT_MAX, np.zeros(4)
    for drop in range(4):
        if comp[drop]:
            continue
        idx = [i for i in range(4) if i != drop]
        sub = s2d(*(S[i] for i in idx))
        x = sum(sub[j] * S[idx[j]] for j in range(3))
        if x @ x < dmin:
            dmin, lam = x @ x, np.zeros(4)
            lam[idx] = sub
    return lam


def box_support(pos, rot, size, dir):
    return rot @ (np.sign(rot.T @ dir) * size) + pos


def gjk_tetra(A, B, iterations=35):
    """GJK on two boxes; returns the final simplex (4 points when they overlap)."""
    x_k = A[0] - B[0]
    simplex = []
    for _ in range(iterations):
        d = x_k / np.linalg.norm(x_k)
        s = box_support(*A, -d) - box_support(*B, d)
        if x_k @ (x_k - s) < 0.0:
            break
        simplex.append(s)
        n = len(simplex)
        lam = {1: lambda: np.ones(1), 2: lambda: s1d(*simplex), 3: lambda: s2d(*simplex),
               4: lambda: s3d(*simplex)}[n]()
        simplex = [simplex[i] for i in range(n) if lam[i] != 0.0]
        x_k = sum(l * p for l, p in zip(lam[lam != 0.0], simplex))
        if len(simplex) == 4:
            break
    return np.array(simplex)


# ---- EPA, shaped like _epa / _attach_face / _add_edge ----
HORIZON_MAX = 24                                         # MJ_MAX_EPAHORIZON in types.py


class Polytope:
    def __init__(self, verts):
        self.vert = [v for v in verts]
        self.center = np.mean(verts, axis=0)
        self.face, self.face_pr, self.face_norm2 = [], [], []
        self.deleted, self.invalid = [], []

    def attach_face(self, v1, v2, v3):
        p1, p2, p3 = self.vert[v1], self.vert[v2], self.vert[v3]
        n = np.cross(p2 - p1, p3 - p1)
        r = (n @ p1) / (n @ n) * n                       # origin projected onto the plane
        if r @ (p1 - self.center) < 0.0:
            r = -r                                       # point outward from the polytope
        self.face.append((v1, v2, v3))
        self.face_pr.append(r)
        self.face_norm2.append(float(r @ r))
        self.deleted.append(False)
        self.invalid.append(False)
        return self.face_norm2[-1]


def add_edge(horizon, e1, e2):
    """_add_edge: an edge seen twice is interior to the deleted region and cancels."""
    edge = (min(e1, e2), max(e1, e2))
    if edge in horizon:
        horizon.remove(edge)
        return True
    if len(horizon) == HORIZON_MAX:
        return False                                     # EPA_HORIZON overflow in MJWarp
    horizon.append(edge)
    return True


def epa(A, B, simplex, iterations=16, tol=1e-7):
    pt = Polytope(simplex)
    for f in [(0, 1, 2), (0, 3, 1), (0, 2, 3), (3, 2, 1)]:          # _polytope4 faces
        pt.attach_face(*f)
    upper, upper2, idx, log = np.inf, np.inf, -1, []
    for k in range(iterations):
        alive = [i for i in range(len(pt.face)) if not (pt.deleted[i] or pt.invalid[i])]
        idx = min(alive, key=lambda i: pt.face_norm2[i])  # closest face: lower bound
        lower2 = pt.face_norm2[idx]
        lower = np.sqrt(lower2)
        normal = pt.face_pr[idx] / lower
        w = box_support(*A, normal) - box_support(*B, -normal)       # _epa_support
        upper = min(upper, float(pt.face_pr[idx] @ w) / lower)
        upper2 = upper * upper
        entry = {"k": k, "lower": round(lower, 6), "upper": round(upper, 6),
                 "closest": idx, "n": normal.round(4).tolist(), "w": w.round(4).tolist(),
                 "verts": [v.round(4).tolist() for v in pt.vert],
                 "faces": [list(pt.face[i]) for i in range(len(pt.face)) if not pt.deleted[i]],
                 "nvalid": len(alive)}
        if upper - lower < tol:
            entry["event"] = "converged"
            log.append(entry)
            break
        if any(np.allclose(w, v) for v in pt.vert):
            entry["event"] = "repeated support point"
            log.append(entry)
            break
        wi = len(pt.vert)
        pt.vert.append(w)
        horizon, removed = [], []
        for i in range(len(pt.face)):                    # every face w can see (+ closest)
            if pt.deleted[i]:
                continue
            if i == idx or pt.face_pr[i] @ w - pt.face_norm2[i] > 1e-10:
                pt.deleted[i] = True
                removed.append(list(pt.face[i]))
                a, b, c = pt.face[i]
                for e in ((a, b), (b, c), (c, a)):
                    assert add_edge(horizon, *e), "horizon overflow"
        added = []
        for e1, e2 in horizon:                           # one new face per horizon edge
            d2 = pt.attach_face(wi, e1, e2)
            if not (lower2 <= d2 <= upper2):
                pt.invalid[-1] = True                    # cannot be the answer: skip it
            added.append([wi, e1, e2])
        entry.update({"event": "expand", "removed": removed, "horizon": horizon,
                      "added": added, "nhorizon": len(horizon)})
        log.append(entry)
    return lower, normal, log


BOXES = """
<mujoco model="box_on_box">
  <worldbody>
    <geom name="A" type="box" size="0.2 0.15 0.05"/>
    <body name="B" pos="0.05 0.02 0.139"><freejoint/>
      <geom name="B" type="box" size="0.1 0.08 0.1"/></body>
  </worldbody>
</mujoco>
"""


def quat(axis, deg):
    a = np.radians(deg) / 2
    return np.concatenate([[np.cos(a)], np.sin(a) * np.asarray(axis) / np.linalg.norm(axis)])


# w0: axis-aligned, B sinks 11 mm into A. w1: B tilted, a corner digs into A's top.
CASES = [("w0 axis-aligned", [0.05, 0.02, 0.139], [1, 0, 0, 0]),
         ("w1 tilted", [-0.093, -0.007, 0.15], quat([0.2, -1.61, 1.81], 44.3))]
mjm = mujoco.MjModel.from_xml_string(BOXES)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=len(CASES), nconmax=8, njmax=32)
qpos = np.array([np.concatenate([p, q]) for _, p, q in CASES])
d.qpos.assign(qpos.astype(np.float32))
mjw.kinematics(m, d)
mjw.collision(m, d)
n = min(int(d.nacon.numpy()[0]), d.naconmax)
cw, cdist = d.contact.worldid.numpy()[:n], d.contact.dist.numpy()[:n]
cframe = d.contact.frame.numpy()[:n]
print("analytic overlap of w0 along z = 0.05 + 0.1 - 0.139 =", round(0.05 + 0.1 - 0.139, 4))

dump = {"cases": []}
for w, (name, p, q) in enumerate(CASES):
    rot = np.zeros(9)
    mujoco.mju_quat2Mat(rot, np.asarray(q, float))
    A = (np.zeros(3), np.eye(3), mjm.geom_size[0])
    B = (np.array(p), rot.reshape(3, 3), mjm.geom_size[1])
    simplex = gjk_tetra(A, B)
    depth, normal, log = epa(A, B, simplex)
    print(f"--- {name}: GJK simplex has {len(simplex)} points")
    for e in log:
        extra = (f"  removed {len(e['removed'])} faces, horizon {e['nhorizon']} edges,"
                 f" +{len(e['added'])} faces" if e["event"] == "expand" else "")
        print(f"  k={e['k']}  faces={len(e['faces']):2d} (valid {e['nvalid']:2d})  "
              f"lower={e['lower']:.6f}  "
              f"upper={e['upper']:.6f}  {e['event']}{extra}")
    sel = cw == w
    print(f"  teaching EPA: depth = {depth:.5f}  normal = {normal.round(4)}")
    print(f"  MJWarp: {sel.sum()} contacts, dist = {[round(float(v), 5) for v in cdist[sel]]},"
          f" normal = {cframe[sel][0, 0].round(4)}")
    dump["cases"].append({"name": name, "A": [A[0].tolist(), A[2].tolist()],
                          "B": [list(p), rot.round(5).tolist(), B[2].tolist()],
                          "simplex": simplex.round(4).tolist(), "depth": round(depth, 6),
                          "log": log})
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 5. Kernels, scratch and tuning
`ccd_iterations` (default 35) bounds GJK and, unless the model is box-box only, EPA; it is compiled into the CCD kernels. Each EPA/multi-contact pair takes a scratch row (`naccdmax` rows per pair-type kernel); running out sets `OverflowType.CCD`.

In [ ]:
# Example 17.7 - ex7_ccd_iterations.py
"""Example 17.7 - how many GJK/EPA iterations does a deep cylinder-box contact need?

A cylinder (r = 0.05) lies along x on the top edge of a box that is turned 45 degrees, so the
edge points up. Where they touch, A - B is curved (the cylinder's round side), so EPA must
approximate a curved surface with flat faces: more iterations, smaller error. The exact depth
is r - |(y, z) of the axis relative to the edge line|. Each world uses a different depth.
MULTICCD is disabled so each pair writes the single EPA contact (multi-contact would replace
the depth with clipped polygon distances, Example 17.9). We rebuild the model with
opt.ccd_iterations = 4, 8 and 35 (the default) using io.override_model, which edits the
model in place and returns None. ccd_iterations bounds both
the GJK loop and, for any model that is not box-box only, the EPA loop; it is a compile-time
constant of the CCD kernel, so each value builds a new kernel.
Run: python ex7_ccd_iterations.py [--dump-json PATH]
  --dump-json PATH writes depth and error per ccd_iterations value for the page's error chart.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.io import override_model  # private API

np.set_printoptions(precision=4, suppress=True)
parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

CYL_ON_EDGE = """
<mujoco model="cyl_on_edge">
  <option><flag multiccd="disable"/></option>
  <worldbody>
    <geom name="ridge" type="box" size="0.3 0.1 0.1" euler="45 0 0"/>
    <body name="cyl" pos="0 0 0.2"><freejoint/>
      <geom name="cyl" type="cylinder" size="0.05 0.1"/></body>
  </worldbody>
</mujoco>
"""
R = 0.05
EDGE_Z = 0.1 * np.sqrt(2)                # the ridge's top edge: the line y = 0, z = EDGE_Z
DEPTHS = [0.005, 0.015, 0.03]            # wanted penetration, one per world
Y_OFF = 0.01                             # axis slightly off the edge: normal is not vertical

mjm = mujoco.MjModel.from_xml_string(CYL_ON_EDGE)
qpos = np.tile(mjm.qpos0, (len(DEPTHS), 1))
for w, depth in enumerate(DEPTHS):
    dz = np.sqrt((R - depth) ** 2 - Y_OFF ** 2)  # axis-to-edge distance is R - depth
    qpos[w, :3] = [0.0, Y_OFF, EDGE_Z + dz]
    qpos[w, 3:7] = [np.cos(np.pi / 4), 0, np.sin(np.pi / 4), 0]   # axis along x
exact = np.array(DEPTHS)

print("exact depth per world:", DEPTHS)
dump = {"exact": DEPTHS, "runs": []}
for iters in [4, 8, 35]:
    m = mjw.put_model(mjm)
    ret = override_model(m, [f"opt.ccd_iterations={iters}"])
    d = mjw.make_data(mjm, nworld=len(DEPTHS), nconmax=8, njmax=48)
    d.qpos.assign(qpos.astype(np.float32))
    mjw.kinematics(m, d)
    mjw.collision(m, d)
    n = min(int(d.nacon.numpy()[0]), d.naconmax)
    wid, dist = d.contact.worldid.numpy()[:n], d.contact.dist.numpy()[:n]
    deepest = np.array([dist[wid == w].min() if (wid == w).any() else np.nan
                        for w in range(len(DEPTHS))])
    err = np.abs(-deepest - exact)
    print(f"ccd_iterations={m.opt.ccd_iterations:2d} (override_model returned {ret})  "
          f"contacts/world={[int((wid == w).sum()) for w in range(len(DEPTHS))]}  "
          f"depth={(-deepest).round(5)}  |error|={err.round(6)}")
    dump["runs"].append({"ccd_iterations": iters,
                         "depth": [None if np.isnan(v) else round(float(-v), 6) for v in deepest],
                         "error": [None if np.isnan(v) else round(float(v), 6) for v in err]})
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Example 17.8 - ex8_naccdmax_overflow.py
"""Example 17.8 - CCD scratch: the naccdmax budget, its overflow bit, and its size in bytes.

Every convex pair that needs EPA (or multi-contact) reserves one row of scratch memory with
ccdid = wp.atomic_add(nccd, geomgeomid, 1). If ccdid >= naccdmax the pair is dropped and the
CCD bit is set in d.overflow (collision_convex.py::ccd_kernel_builder). Part 1 puts 6
ellipsoids into 6 boxes in each of 2 worlds (12 EPA pairs) and runs with nccdmax = 2 per world
(make_data multiplies it by nworld: naccdmax = 4) and with the default (naccdmax = naconmax). Part 2 computes the per-row scratch that
convex_narrowphase allocates in 4.0.1 from the same rules, and checks the rules by recording
the real wp.empty calls of collision_convex.py for four small models.
Run: python ex8_naccdmax_overflow.py [--dump-json PATH]
  --dump-json PATH writes the measured scratch sizes for the page's scratch calculator.
"""
import argparse
import json
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.collision_driver import MJ_COLLISION_TABLE  # private API

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 2
MJ_MAX_EPAHORIZON, MJ_MAX_EPAFACES = 24, 5          # constants from _src/types.py (4.0.1)


def ellipsoids_in_boxes(k=6):
    parts = []
    for i in range(k):
        x = 0.3 * i
        parts.append(f'<geom type="box" pos="{x} 0 0.05" size="0.1 0.1 0.05"/>'
                     f'<body pos="{x} 0 0.12"><freejoint/>'
                     f'<geom type="ellipsoid" size="0.06 0.05 0.04"/></body>')
    return f'<mujoco><worldbody>{"".join(parts)}</worldbody></mujoco>'


# ---------- Part 1: overflow ----------
dump = {"part1": [], "rows": []}
mjm = mujoco.MjModel.from_xml_string(ellipsoids_in_boxes())
for nccdmax in [2, None]:                             # per world, like nconmax; None = default
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0                           # keep stdout clean; the bit is still set
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64, nccdmax=nccdmax)
    mjw.kinematics(m, d)
    mjw.collision(m, d)
    n = min(int(d.nacon.numpy()[0]), d.naconmax)
    ov = d.overflow.numpy()
    bits = [mjw.OverflowType(int(v)) for v in ov]
    print(f"naccdmax={d.naccdmax:3d} (naconmax={d.naconmax}): contacts written = {n} of "
          f"{6 * NWORLD} penetrating pairs; overflow per world = {[b.name for b in bits]}")
    worlds = [int(w) for w in d.contact.worldid.numpy()[:n]]
    print(f"  worldid of the written contacts = {worlds}")
    dump["part1"].append({"naccdmax": d.naccdmax, "naconmax": d.naconmax, "worlds": worlds,
                          "pairs": 6 * NWORLD, "overflow": [b.name for b in bits]})

# ---------- Part 2: scratch bytes per CCD row ----------
VEC3, INT, FLT = 12, 4, 4


def rules(epa_iterations, npolygonmax, nmeshdegmax):
    """Bytes per row of every array convex_narrowphase allocates (4.0.1, lines ~1311-1438)."""
    nv, nf = 10 + 2 * epa_iterations, 6 + MJ_MAX_EPAFACES * epa_iterations
    epa = {"epa_vert": nv * VEC3, "epa_vert_index": nv * INT, "epa_face": nf * INT,
           "epa_pr": nf * VEC3, "epa_norm2": nf * FLT, "epa_horizon": MJ_MAX_EPAHORIZON * INT}
    p, q = npolygonmax, nmeshdegmax
    multi = {"multiccd_polygon": 2 * p * VEC3, "multiccd_clipped": 2 * p * VEC3,
             "multiccd_pnormal": p * VEC3, "multiccd_pdist": p * FLT,
             "multiccd_idx1": q * INT, "multiccd_idx2": q * INT, "multiccd_n1": q * VEC3,
             "multiccd_n2": q * VEC3, "multiccd_endvert": q * VEC3,
             "multiccd_face1": p * VEC3, "multiccd_face2": p * VEC3}
    return epa, multi


def predicted_dims(m):
    """The epa_iterations / npolygonmax / nmeshdegmax rules of convex_narrowphase."""
    G = mjw.GeomType
    ng = len(G)

    def count(a, b):
        a, b = min(a, b), max(a, b)
        return m.geom_pair_type_count[a * ng - a * (a + 1) // 2 + b]   # upper_trid_index
    convex = [k for k, v in MJ_COLLISION_TABLE.items() if v.name == "CONVEX"]
    total = sum(count(int(a), int(b)) for a, b in convex)
    nboxbox = count(G.BOX, G.BOX)
    epa_it = 16 if nboxbox == total else m.opt.ccd_iterations
    multi = (m.opt.disableflags & mjw.DisableBit.MULTICCD) == 0
    P, Q = (4, 3) if nboxbox else (0, 0)
    if multi and count(5, 5) + count(5, 6) + count(5, 7) + count(3, 5) > 0:
        P, Q = max(m.npolygonmax, 16), max(m.nmeshdegmax, 3)
    elif multi and count(7, 7) + count(6, 7) + count(3, 7) > 0:
        P, Q = max(m.npolygonmax, 4 if count(6, 7) else P), max(m.nmeshdegmax, 3)
    return epa_it, P, Q


def record_allocs(xml):
    """Run collision once and record every wp.empty made inside collision_convex.py."""
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=1, nconmax=16, njmax=64)
    mjw.kinematics(m, d)
    seen, orig = [], wp.empty

    def spy(*a, **k):
        arr = orig(*a, **k)
        if sys._getframe(1).f_code.co_filename.endswith("collision_convex.py"):
            seen.append(arr)
        return arr
    wp.empty = spy
    try:
        mjw.collision(m, d)
    finally:
        wp.empty = orig
    rows = [a for a in seen if a.ndim == 2 and a.shape[0] == d.naccdmax]
    measured = sum(a.shape[1] * wp.types.type_size_in_bytes(a.dtype) for a in rows)
    return m, d, measured


MESH = ('<asset><mesh name="hex" vertex="' + " ".join(
    f"{0.05 * np.cos(a):.4f} {0.05 * np.sin(a):.4f} {z}" for z in (-.03, .03)
    for a in np.arange(6) * np.pi / 3) + '"/></asset>')
BOX = '<body pos="{x} 0 0.15"><freejoint/><geom type="box" size=".05 .05 .05"/></body>'
CONFIGS = {
    "ellipsoid-box (Part 1)": ellipsoids_in_boxes(),
    "box-box only": f'<mujoco><worldbody><geom type="box" size=".1 .1 .1"/>'
                    f'{BOX.format(x=0)}</worldbody></mujoco>',
    "+ cylinder": f'<mujoco><worldbody><geom type="box" size=".1 .1 .1"/>{BOX.format(x=0)}'
                  f'<body pos="1 0 .1"><freejoint/><geom type="cylinder" size=".05 .05"/></body>'
                  f'</worldbody></mujoco>',
    "+ mesh": f'<mujoco>{MESH}<worldbody><geom type="box" size=".1 .1 .1"/>{BOX.format(x=0)}'
              f'<body pos="1 0 .1"><freejoint/><geom type="mesh" mesh="hex"/></body>'
              f'</worldbody></mujoco>',
}
print(f"{'model':24s} {'epa_it':>6s} {'npoly':>5s} {'ndeg':>4s} {'EPA B':>6s} {'multi B':>7s}"
      f" {'rule B/row':>10s} {'measured':>8s} {'naccdmax':>8s} {'total':>9s}")
for name, xml in CONFIGS.items():
    m, d, measured = record_allocs(xml)
    e_it, P, Q = predicted_dims(m)
    epa, multi = rules(e_it, P, Q)
    rule = sum(epa.values()) + sum(multi.values())
    print(f"{name:24s} {e_it:6d} {P:5d} {Q:4d} {sum(epa.values()):6d} {sum(multi.values()):7d}"
          f" {rule:10d} {measured:8d} {d.naccdmax:8d} {rule * d.naccdmax:9d}")
    dump["rows"].append({"name": name, "epa_iterations": e_it, "npolygonmax": P,
                         "nmeshdegmax": Q, "epa_bytes": sum(epa.values()),
                         "multi_bytes": sum(multi.values()), "measured": measured,
                         "naccdmax": d.naccdmax, "epa": epa, "multi": multi})
nslot = 4096 * 48 if ON_GPU else 8 * 48
print(f"e.g. a mesh scene with naccdmax = {nslot}: "
      f"{dump['rows'][-1]['measured'] * nslot / 2**20:.1f} MiB of CCD scratch per collision call")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 6. Multi-contact
Aligned faces are clipped against each other (`_polygon_clip`) and reduced to 4 points (`_polygon_quad`). Always on for box-box; other face/face and edge/face pairs follow `DisableBit.MULTICCD`.

In [ ]:
# Example 17.9 - ex9_multiccd.py
"""Example 17.9 - multi-contact: one EPA witness pair becomes up to 4 contacts.

GJK/EPA returns ONE pair of witness points. A box resting flat on a box needs several contacts
or it rocks. collision_gjk.py::multicontact finds the two aligned faces, clips one against the
other (_polygon_clip) and keeps at most 4 points (_polygon_quad). It runs for face/face and
face/edge pairs, unless DisableBit.MULTICCD is set; box-box multi-contact is always on
(ccd_kernel_builder checks geomtype1 == geomtype2 == BOX before use_multiccd).
We count contacts for box-on-box, cylinder-on-box and mesh-on-box, flat and tilted 20 degrees
about a horizontal axis, with and without MULTICCD.
Run: python ex9_multiccd.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

ON_GPU = wp.get_device().is_cuda

HEX = " ".join(f"{0.05 * np.cos(a):.5f} {0.05 * np.sin(a):.5f} {z}"
               for z in (-0.03, 0.03) for a in np.arange(6) * np.pi / 3)


def scene(multiccd):
    flag = "" if multiccd else '<option><flag multiccd="disable"/></option>'
    return f"""
<mujoco model="multiccd">
  {flag}
  <asset><mesh name="hex" vertex="{HEX}"/></asset>
  <worldbody>
    <geom name="t0" type="box" pos="0 0 0.05" size="0.1 0.1 0.05"/>
    <geom name="t1" type="box" pos="1 0 0.05" size="0.1 0.1 0.05"/>
    <geom name="t2" type="box" pos="2 0 0.05" size="0.1 0.1 0.05"/>
    <body name="box" pos="0 0 0.149"><freejoint/><geom type="box" size="0.05 0.05 0.05"/></body>
    <body name="cyl" pos="1 0 0.149"><freejoint/><geom type="cylinder" size="0.05 0.05"/></body>
    <body name="hex" pos="2 0 0.129"><freejoint/><geom type="mesh" mesh="hex"/></body>
  </worldbody>
</mujoco>"""


def tilted_qpos(mjm):
    """World 1: every body turned 20 deg about the y-axis, lowest point 1 mm deep."""
    q = mjm.qpos0.copy()
    t = np.radians(20.0)
    for b, (r, h) in enumerate([(0.05, 0.05), (0.05, 0.05), (0.05, 0.03)]):
        adr = 7 * b
        low = h * np.cos(t) + r * np.sin(t)               # centre height above the lowest point
        q[adr + 2] = 0.1 + low - 0.001
        q[adr + 3:adr + 7] = [np.cos(t / 2), 0, np.sin(t / 2), 0]
    return q


labels = ["box-box", "cylinder-box", "mesh-box"]
for multiccd in (True, False):
    mjm = mujoco.MjModel.from_xml_string(scene(multiccd))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=2, nconmax=24, njmax=96)
    d.qpos.assign(np.stack([mjm.qpos0, tilted_qpos(mjm)]).astype(np.float32))
    mjw.kinematics(m, d)
    mjw.collision(m, d)
    n = min(int(d.nacon.numpy()[0]), d.naconmax)
    geom, wid = d.contact.geom.numpy()[:n], d.contact.worldid.numpy()[:n]
    print(f"MULTICCD {'enabled ' if multiccd else 'disabled'}:")
    for w, pose in enumerate(["flat  ", "tilted"]):
        counts = []
        for k in range(3):
            body_geom = 3 + k                              # geoms 3, 4, 5 are the free bodies
            counts.append(int(((wid == w) & ((geom == body_geom).any(axis=1))).sum()))
        print(f"  {pose}: " + "  ".join(f"{lab} {c}" for lab, c in zip(labels, counts)))
    print("  overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 7. Meshes collide as their convex hull
The mesh support function returns the best vertex (hill-climbing on the hull graph), which is the support of the convex hull.

In [ ]:
# Example 17.10 - ex10_nonconvex_mesh.py
"""Example 17.10 - a non-convex mesh collides as its convex hull.

The support function of a mesh (collision_gjk.py::support, MESH branch) returns the vertex
with the largest dot(v, dir): that is the support function of the CONVEX HULL of the
vertices, so GJK/EPA only ever see the hull. An L-shaped block (seen from above) has a notch;
a ball dropped into the notch should fall to the floor, but in MJWarp (and MuJoCo C) it rests on
the invisible hull lid at the block's top height. We print the ball height over time and the
expected heights for "hull" and "true L shape".
Run: python ex10_nonconvex_mesh.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4

S, T, H = 0.4, 0.12, 0.1      # L arm length, arm width, block height
L2D = [(0, 0), (S, 0), (S, T), (T, T), (T, S), (0, S)]   # L footprint, notch at (S, S)
VERT = " ".join(f"{x} {y} {z}" for z in (0.0, H) for x, y in L2D)
R = 0.05
NOTCH = (0.2, 0.2)            # inside the hull (x + y < S + T), outside the L

L_MESH = f"""
<mujoco model="l_block">
  <option timestep="0.002"/>
  <asset><mesh name="L" vertex="{VERT}"/></asset>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="2 2 .1"/>
    <geom name="L" type="mesh" mesh="L"/>
    <body name="ball" pos="{NOTCH[0]} {NOTCH[1]} 0.35"><freejoint/>
      <geom name="ball" type="sphere" size="{R}"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(L_MESH)
print(f"mesh: {mjm.mesh_vertnum[0]} vertices; MuJoCo stores the convex hull as the collision "
      f"graph (mesh_graphadr = {mjm.mesh_graphadr[0]})")
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
lz = mjm.geom_pos[1][2]                        # MuJoCo re-centres the mesh at its centroid
top = lz + H / 2
print(f"block top at z = {top:.4f}; ball centre would rest at {top + R:.4f} on the hull lid "
      f"and at {R:.4f} on the floor of a truly L-shaped block")
for step in range(500):
    mjw.step(m, d)
    if step % 100 == 99:
        z = d.qpos.numpy()[:, 2]
        print(f"t = {(step + 1) * mjm.opt.timestep:.1f} s  ball z (all worlds) = {z.round(4)}")
n = min(int(d.nacon.numpy()[0]), d.naconmax)
pairs = {tuple(int(g) for g in gp) for gp in d.contact.geom.numpy()[:n]}
names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(mjm.ngeom)]
print("touching pairs:", sorted(f"{names[a]}-{names[b]}" for a, b in pairs))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## Exercise 17.2: a cylinder support function
Mirror the CYLINDER branch of `support()` (margin 0) and check it on 8 directions against 40,000 rim samples.

In [ ]:
# Exercise 17.2: a cylinder support function
import numpy as np

MINVAL = 1e-15


def cylinder_support(size, pos, rot, dir):
    ...   # rotate dir into the local frame, rim point in xy, +-half-height by the sign of z


rng = np.random.default_rng(1)
size = np.array([0.05, 0.08, 0.0])
a = 0.4
rot = np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])
pos = np.array([0.1, -0.2, 0.3])
ang = rng.uniform(0, 2 * np.pi, 40000)
z = rng.choice([-1.0, 1.0], 40000) * size[1]
rim = np.column_stack([size[0] * np.cos(ang), size[0] * np.sin(ang), z]) @ rot.T + pos
dirs = [np.array(v, float) for v in ([1, 0, 0], [0, 1, 0], [0, 0, 1], [0, 0, -1],
        [1, 1, 1], [-1, 2, 0.5], [0.3, -0.2, -1], [-1, -1, 0.2])]
if cylinder_support(size, pos, rot, dirs[0]) is None:
    print("Exercise 17.2: fill in cylinder_support, then re-run this cell")
else:
    for dvec in dirs:
        dn = dvec / np.linalg.norm(dvec)
        s = cylinder_support(size, pos, rot, dvec)
        assert abs((rim @ dn).max() - s @ dn) < 1e-6, dvec
    print("8 directions pass")

In [ ]:
#@title Exercise 17.2: solution (double-click to show the code)
import numpy as np

MINVAL = 1e-15


def cylinder_support(size, pos, rot, dir):
    """Mirror of the CYLINDER branch of collision_gjk.py::support (margin 0)."""
    local = rot.T @ (dir / np.linalg.norm(dir))
    res = np.zeros(3)
    d = np.hypot(local[0], local[1])
    if d > MINVAL:                                  # point on the rim circle
        res[:2] = local[:2] * size[0] / d
    res[2] = size[1] if local[2] >= 0.0 else -size[1]   # top or bottom cap
    return rot @ res + pos


rng = np.random.default_rng(1)
size = np.array([0.05, 0.08, 0.0])
a = 0.4
rot = np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])
pos = np.array([0.1, -0.2, 0.3])
ang = rng.uniform(0, 2 * np.pi, 40000)
z = rng.choice([-1.0, 1.0], 40000) * size[1]
rim = np.column_stack([size[0] * np.cos(ang), size[0] * np.sin(ang), z]) @ rot.T + pos
dirs = [np.array(v, float) for v in ([1, 0, 0], [0, 1, 0], [0, 0, 1], [0, 0, -1],
        [1, 1, 1], [-1, 2, 0.5], [0.3, -0.2, -1], [-1, -1, 0.2])]
for dvec in dirs:
    dn = dvec / np.linalg.norm(dvec)
    s = cylinder_support(size, pos, rot, dvec)
    assert abs((rim @ dn).max() - s @ dn) < 1e-6, dvec
print("8 directions pass")

## Stretch: what does CCD cost?
On a GPU runtime, time `mjw.collision` alone for a primitive-only scene and a convex scene with the same contacts per world, for several `nworld`, then vary `ccd_iterations` and `naccdmax`.

In [ ]:
# Stretch (Colab GPU): what does CCD cost? A starting point; report only your own numbers.
import time

def time_collision(m, d, n=20):
    mjw.kinematics(m, d)
    mjw.collision(m, d)           # warm-up (JIT)
    wp.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        mjw.collision(m, d)
    wp.synchronize()
    return (time.perf_counter() - t0) / n

# Build (a) a primitive-only scene and (b) a convex scene with the same contacts per world,
# then compare d.nworld / time_collision(m, d) for nworld = 1024, 4096, 16384 (GPU only),
# and for scene (b) also ccd_iterations 8 vs 35 and naccdmax "just enough" vs the default.
print("GPU available:", ON_GPU)